In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from math import comb, pi, sqrt

from sklearn import svm
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score,accuracy_score

# these imports are from three_profile.py and four_profile.py files
from four_profile_src.four_profile import four_profile
from four_profile_src.three_profile import three_profile

In [2]:
df_oil_k3 = pd.read_csv('data/Oil Trade k3 graph.csv')

In [3]:
def train_svm(X_train,y_train, kfold):
    svc_clasf = svm.SVC(max_iter=10000)
    svc_grid_search = GridSearchCV(
        svc_clasf,
        param_grid = {
            'C' : [0.05, 0.1, 0.5, 1, 5, 10, 500, 1000],
            'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
            'degree': [2,3,4]
        },
        verbose=0,
        refit=True,
        scoring='f1_weighted', cv=kfold)
    svc_grid_search.fit(X_train, y_train)

    svc_bclasf = svc_grid_search.best_estimator_
    svc_bclasf.fit(X_train,y_train)
    # svc_y_pred = svc_bclasf.predict(X_test)
    # print("SVC: \n",
    #     "Accuracy:", accuracy_score(y_test, svc_y_pred), "\n",
    #     "F1-Score:", f1_score(y_test,svc_y_pred, average='weighted'), "\n",
    #     "Oil Trade Prediction:", svc_bclasf.predict([graph_embedding])
    # )

    return svc_bclasf

In [4]:
def train_decision_tree(X_train, y_train, kfold):
    dt_clasf = DecisionTreeClassifier(random_state=42)
    dt_gridsearch = GridSearchCV(
        dt_clasf,
        param_grid = {
            'splitter' : ['best'],
            'criterion' : ['entropy','gini'],
            'max_features' : [0.2,'sqrt',1.],
            'max_depth' : [2,4],
            'class_weight' : ['balanced']
        },
        verbose=0,
        refit=True,
        cv=kfold,
        scoring='f1_weighted'
    )
    dt_gridsearch.fit(X_train,y_train)

    dt_bclasf = dt_gridsearch.best_estimator_
    dt_bclasf.fit(X_train,y_train)

    # dt_y_pred = dt_bclasf.predict(X_test)
    # print("Decision Tree: \n",
    #     "Accuracy:", accuracy_score(y_test, dt_y_pred), "\n",
    #     "F1-Score:", f1_score(y_test,dt_y_pred, average='weighted'), "\n",
    #     "Oil Trade Prediction:", dt_bclasf.predict([graph_embedding])
    # )

    return dt_bclasf

In [5]:
def train_random_forest(X_train, y_train, kfold):
    rf_clasf = RandomForestClassifier(random_state=42)
    rf_gridsearch = GridSearchCV(
        rf_clasf,
        param_grid = {
            'n_estimators' : [5,10,25,50,100],
            'criterion' : ['entropy','gini'],
            'max_features' : [0.2,'sqrt',1.],
            'max_depth' : [2,4],
            'class_weight' : ['balanced'],
        },
        verbose=0,
        refit=True,
        cv=kfold,
        scoring='f1_weighted'
    )
    rf_gridsearch.fit(X_train,y_train)

    rf_bclasf = rf_gridsearch.best_estimator_
    rf_bclasf.fit(X_train,y_train)

    # rf_y_pred = rf_bclasf.predict(X_test)
    # print("Random Forest: \n",
    #     "Accuracy:", accuracy_score(y_test, rf_y_pred), "\n",
    #     "F1-Score:", f1_score(y_test,rf_y_pred, average='weighted'), "\n",
    #     "Oil Trade Prediction:", rf_bclasf.predict([graph_embedding])
    # )
    return rf_bclasf

In [6]:
def train_ada_boost(X_train, y_train, kfold):
    ada_clasf = AdaBoostClassifier(random_state=42)
    ada_gridsearch = GridSearchCV(
        ada_clasf,
        param_grid = {
            'n_estimators' : [5,10,25,50,100],
            'learning_rate' : [0.1,0.3,0.5]
            },
        verbose=0,
        refit=True,
        cv=kfold,
        scoring='f1_weighted',
    )
    ada_gridsearch.fit(X_train,y_train)

    ada_bclasf = ada_gridsearch.best_estimator_
    ada_bclasf.fit(X_train,y_train)

    # ada_y_pred = ada_bclasf.predict(X_test)
    # print("Adaboost: \n",
    #     "Accuracy:", accuracy_score(y_test, ada_y_pred), "\n",
    #     "F1-Score:", f1_score(y_test,ada_y_pred, average='weighted'), "\n",
    #     "Oil Trade Prediction:", ada_bclasf.predict([graph_embedding])
    # )
    return ada_bclasf


In [7]:
k3_oil_graphs = {}
svm_chunglu_predictions = {}
rf_chunglu_predictions = {}
ada_chunglu_predictions = {}
tree_predictions = {}
# train graph models for every year to check chunglu percentage
for year in range(1988,2024):
    k3_oil_graph = nx.Graph()
    for row in df_oil_k3.itertuples():
        if row.year == year:
            k3_oil_graph.add_edge(row.source, row.target, year=row.year)

    n = k3_oil_graph.number_of_nodes()
    m = k3_oil_graph.number_of_edges()

    # gnp generator
    p = m / comb(n, 2)
    gnp_graphs = []
    for i in range(100):
        gnp = nx.gnp_random_graph(n, p, seed=i)
        gnp_graphs.append(gnp)

    # ChungLu Generator
    degree_distribution = [d for v, d in k3_oil_graph.degree()]
    chunglu_graphs = []
    for i in range(100):
        g_chunglu = nx.expected_degree_graph(degree_distribution, seed=i, selfloops=False)
        chunglu_graphs.append(g_chunglu)

    # PA, preferential attachment
    degree_distribution = [d for v,d in k3_oil_graph.degree()]
    pa_m = (m - 1) / n # m parameter for PA, 2/n + 2m = 2|E| / n

    pa_graphs = []
    for i in range(100):
        g_pa = nx.barabasi_albert_graph(n=n, m=int(pa_m), seed=i)
        pa_graphs.append(g_pa)

    # Configuration (random graph with same degree distribution)
    degree_distribution = [d for v,d in k3_oil_graph.degree()]
    config_graphs = []
    for i in range(100):
        g_config = nx.configuration_model(degree_distribution, seed=i)
        g_config.remove_edges_from(nx.selfloop_edges(g_config))
        g_config_simple = nx.Graph(g_config)
        config_graphs.append(g_config_simple)

    # Geometric Model
    r = sqrt((2.0 * m) / (n * (n-1) * pi))
    # r = sqrt((2.0 * m) / (n * (n-1) * pi)) * 1.08
    # typically, r would be set without 1.08 multiplier, however, since # edges is so large, the radius needs to be rather large
    # in the case of large radius, the approximation of r without the 1.08 multipler falls short because the radius
    # of the circles can go past the boundary of the unit cube

    geometric_graphs = []
    for i in range(100):
        g_geometric = nx.random_geometric_graph(n=n, radius=r, seed=i)
        geometric_graphs.append(g_geometric)

    x = np.empty([500,15])
    x[:] = np.nan

    for i in range(100):
        x[i] = np.array(list(four_profile(gnp_graphs[i])) + list(three_profile(gnp_graphs[i])), dtype=np.int64)
        x[i+100] = np.array(list(four_profile(chunglu_graphs[i])) + list(three_profile(chunglu_graphs[i])), dtype=np.int64)
        x[i+200] = np.array(list(four_profile(pa_graphs[i])) + list(three_profile(pa_graphs[i])), dtype=np.int64)
        x[i+300] = np.array(list(four_profile(config_graphs[i])) + list(three_profile(config_graphs[i])), dtype=np.int64)
        x[i+400] = np.array(list(four_profile(geometric_graphs[i])) + list(three_profile(geometric_graphs[i])), dtype=np.int64)

    # single dimension y
    # labels: 1 is gnp, 2 is chunglu, 3 is pa, 4 is config, 5 is geometric
    y = np.empty([500,])
    y[0:100] = 1
    y[100:200] = 2
    y[200:300] = 3
    y[300:400] = 4
    y[400:500] = 5

    kfold = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)
    graph_embedding = np.array(list(four_profile(k3_oil_graph)) + list(three_profile(k3_oil_graph)), dtype=np.int64)

    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=17, stratify=y)

    svm_clasf = train_svm(X_train,y_train, kfold)
    tree_clasf = train_decision_tree(X_train, y_train, kfold)
    rf_clasf = train_random_forest(X_train, y_train, kfold)
    ada_clasf = train_ada_boost(X_train, y_train, kfold)

    calibrated_svm_clf = CalibratedClassifierCV(svm_clasf, method='sigmoid')
    calibrated_svm_clf.fit(X_train,y_train)
    svm_predictions = calibrated_svm_clf.predict_proba([graph_embedding])
    svm_chunglu_predictions = svm_predictions[:,1]

    rf_predictions = rf_clasf.predict_proba([graph_embedding])
    rf_chunglu_predictions[year] = rf_predictions[:,1]

    # Adaboost predictions need calibration, can use CalibratedClassifierCV from sklearn
    # See paper: https://www.cs.cornell.edu/~caruana/niculescu.scldbst.crc.rev4.pdf for more details
    # ada_bclasf.predict_proba([graph_embedding])

    calibrated_clf = CalibratedClassifierCV(ada_clasf, method='sigmoid')
    calibrated_clf.fit(X_train,y_train)
    ada_predictions = calibrated_clf.predict_proba([graph_embedding])
    ada_chunglu_predictions = ada_predictions[:,1]

    tree_predictions[year] = tree_clasf.predict([graph_embedding])[0]



C:\Users\vince\Git\tradeNetwork\.venv\Lib\site-packages\sklearn\svm\_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
C:\Users\vince\Git\tradeNetwork\.venv\Lib\site-packages\sklearn\svm\_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
C:\Users\vince\Git\tradeNetwork\.venv\Lib\site-packages\sklearn\svm\_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
C:\Users\vince\Git\tradeNetwork\.venv\Lib\site-packages\sklearn\svm\_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
C:\Users\vince\Git\tradeNetwork\.venv\Lib\site-packages\sklearn\svm\

KeyboardInterrupt: 